# Notebook 08 — Climate Attribution Visualization, Uncertainty & Publication-Ready Reporting

## Purpose
This notebook is the final reporting layer of the climate attribution toolkit. It converts validated outputs from Notebooks 01–07 into publication-ready figures, uncertainty products, summary tables, and reproducible reporting files.

**Workflow:** preprocessing → indicators → trends → bias correction → ensemble → attribution/risk → reporting.

> This notebook reports results; it does not replace formal probabilistic event-attribution methods.

## 1. Objectives
- Standardize figures and tables.
- Visualize ensemble means/medians, model spread and agreement.
- Compare baseline and future periods.
- Report trends, extremes and risk.
- Perform final data/figure/statistical QC.
- Create a traceable results package.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
print("Libraries loaded.")

## 2. Project configuration

In [ ]:
BASE = Path(".")
OUTPUTS = BASE / "outputs"
FIGURES = BASE / "figures" / "publication"
TABLES = BASE / "tables" / "publication"
REPORTS = BASE / "reports"
for p in [OUTPUTS, FIGURES, TABLES, REPORTS]: p.mkdir(parents=True, exist_ok=True)

VARIABLE = "tasmax"
SCENARIO = "rcp26"
BASELINE_START, BASELINE_END = "1981-01-01", "2000-12-31"
FUTURE_START, FUTURE_END = "2081-01-01", "2100-12-31"
FIG_DPI = 300
print(FIGURES, TABLES)

## 3. Reporting conventions

In [ ]:
REPORTING = {
    "temperature_unit": "°C", "precipitation_unit": "mm/day",
    "change_decimals": 2, "trend_decimals": 3, "figure_dpi": FIG_DPI,
}
REPORTING

## 4. File naming convention
Use descriptive names such as `TXx_rcp26_future_ensemble_mean_change.png`, `RX5day_rcp26_model_agreement.png`, and `SPEI3_rcp26_uncertainty.png`.

In [ ]:
def figure_path(name): return FIGURES / f"{name}.png"
def table_path(name): return TABLES / f"{name}.csv"

## 5. Load existing NetCDF outputs

In [ ]:
# Adapt this pattern to your actual output folders.
# ds = xr.open_dataset("outputs/indicators/rcp26/TXx/TXx_ensemble_mean.nc")
# da = ds["TXx"]

## 6. Load a DataArray safely

In [ ]:
def open_variable(path, variable=None):
    ds = xr.open_dataset(path)
    if variable is None:
        variable = list(ds.data_vars)[0]
    return ds[variable]

# Example:
# TXx = open_variable("path/to/file.nc", "TXx")

## 7. Period selection

In [ ]:
def select_period(da, start, end):
    return da.sel(time=slice(start, end))

def period_mean(da, start, end): return select_period(da,start,end).mean("time")
def period_sum(da, start, end): return select_period(da,start,end).sum("time")
def period_max(da, start, end): return select_period(da,start,end).max("time")

## 8. Climate-change signal

In [ ]:
def absolute_change(future, baseline): return future - baseline
def relative_change(future, baseline): return (future - baseline) / baseline * 100

# Example:
# change = absolute_change(future, baseline)

## 9. Ensemble statistics

In [ ]:
def ens_mean(da): return da.mean("model")
def ens_median(da): return da.median("model")
def ens_std(da): return da.std("model")
def ens_range(da): return da.max("model") - da.min("model")
def ens_p10(da): return da.quantile(.10, dim="model")
def ens_p90(da): return da.quantile(.90, dim="model")

## 10. Model agreement

In [ ]:
def increase_fraction(da): return (da > 0).mean("model")
def decrease_fraction(da): return (da < 0).mean("model")
def agreement_percent(da): return increase_fraction(da) * 100

## 11. Robust-signal mask

In [ ]:
def robust_increase(change, threshold=.80):
    return (change > 0).mean("model") >= threshold

def robust_decrease(change, threshold=.80):
    return (change < 0).mean("model") >= threshold

## 12. Model spread

In [ ]:
def p10_p90_spread(da): return ens_p90(da) - ens_p10(da)
def iqr(da): return da.quantile(.75, dim="model") - da.quantile(.25, dim="model")

## 13. Latitude-aware area weighting

In [ ]:
def area_weighted_mean(da):
    lat = "lat" if "lat" in da.coords else "latitude"
    lon = "lon" if "lon" in da.coords else "longitude"
    weights = np.cos(np.deg2rad(da[lat]))
    return da.weighted(weights).mean((lat, lon))

## 14. Regional summary

In [ ]:
def regional_summary(name, baseline, future):
    b = float(np.nanmean(np.asarray(baseline))); f = float(np.nanmean(np.asarray(future)))
    return {"indicator":name, "baseline":b, "future":f, "absolute_change":f-b,
            "relative_change_percent":(f-b)/abs(b)*100 if b else np.nan}

## 15. Standard map function

In [ ]:
def plot_map(da, title, filename, cmap=None, vmin=None, vmax=None, cbar_label="", figsize=(9,6)):
    fig, ax = plt.subplots(figsize=figsize)
    da.plot(ax=ax, cmap=cmap, vmin=vmin, vmax=vmax, cbar_kwargs={"label":cbar_label})
    ax.set_title(title); ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    fig.tight_layout(); fig.savefig(figure_path(filename), dpi=FIG_DPI, bbox_inches="tight"); plt.show(); plt.close(fig)

## 16. Baseline map

In [ ]:
# plot_map(baseline_da, "Baseline TXx, 1981–2000", "TXx_rcp26_baseline", cbar_label="TXx (°C)")

## 17. Future map

In [ ]:
# plot_map(future_da, "Future TXx, 2081–2100", "TXx_rcp26_future", cbar_label="TXx (°C)")

## 18. Absolute-change map

In [ ]:
# change_da = future_da - baseline_da
# vmin, vmax = -abs(change_da).max().item(), abs(change_da).max().item()
# plot_map(change_da, "Projected TXx change", "TXx_rcp26_change", vmin=vmin, vmax=vmax, cbar_label="Change (°C)")

## 19. Relative-change map

In [ ]:
# rel = (future_da - baseline_da) / baseline_da * 100
# plot_map(rel, "Projected relative change", "indicator_rcp26_relative_change", cbar_label="Change (%)")

## 20. Uncertainty map

In [ ]:
# spread = ens_std(model_change_da)
# plot_map(spread, "Inter-model standard deviation", "indicator_rcp26_model_std", cbar_label="Model standard deviation")

## 21. Model-range map

In [ ]:
# spread = ens_range(model_change_da)
# plot_map(spread, "Inter-model range", "indicator_rcp26_model_range", cbar_label="Model range")

## 22. P10–P90 spread map

In [ ]:
# spread = p10_p90_spread(model_change_da)
# plot_map(spread, "P10–P90 model spread", "indicator_rcp26_p10_p90", cbar_label="P90 − P10")

## 23. Agreement map

In [ ]:
def plot_agreement(change, title, filename):
    plot_map(agreement_percent(change), title, filename, vmin=0, vmax=100, cbar_label="Models increasing (%)")

## 24. Agreement categories

In [ ]:
def agreement_category(change):
    p = increase_fraction(change)
    return xr.where(p>=.8, 3, xr.where(p>=.6, 2, xr.where(p>=.4, 1, 0)))
# 0 weak/mixed, 1 moderate, 2 strong, 3 very strong

## 25. Regional time series

In [ ]:
def regional_series(da):
    lat = "lat" if "lat" in da.coords else "latitude"
    lon = "lon" if "lon" in da.coords else "longitude"
    return da.weighted(np.cos(np.deg2rad(da[lat]))).mean((lat,lon))

## 26. Time-series plot

In [ ]:
def plot_timeseries(series, title, ylabel, filename):
    fig, ax = plt.subplots(figsize=(11,5)); series.plot(ax=ax)
    ax.set_title(title); ax.set_xlabel("Time"); ax.set_ylabel(ylabel)
    fig.tight_layout(); fig.savefig(figure_path(filename), dpi=FIG_DPI, bbox_inches="tight"); plt.show(); plt.close(fig)

## 27. Ensemble time series with P10–P90

In [ ]:
def plot_ensemble_timeseries(models, title, ylabel, filename):
    mean=models.mean("model"); p10=models.quantile(.10,dim="model"); p90=models.quantile(.90,dim="model")
    fig,ax=plt.subplots(figsize=(11,5)); mean.plot(ax=ax,label="Ensemble mean")
    ax.fill_between(mean.time.values,p10.values,p90.values,alpha=.25,label="P10–P90 model spread")
    ax.set_title(title); ax.set_xlabel("Time"); ax.set_ylabel(ylabel); ax.legend()
    fig.tight_layout(); fig.savefig(figure_path(filename),dpi=FIG_DPI,bbox_inches="tight"); plt.show(); plt.close(fig)

## 28. Individual-model spaghetti plot

In [ ]:
def plot_models(models, title, ylabel, filename):
    fig,ax=plt.subplots(figsize=(11,5))
    for m in models.model.values: models.sel(model=m).plot(ax=ax,label=str(m),alpha=.65)
    ax.set_title(title); ax.set_xlabel("Time"); ax.set_ylabel(ylabel); ax.legend(bbox_to_anchor=(1.02,1),loc="upper left",fontsize=8)
    fig.tight_layout(); fig.savefig(figure_path(filename),dpi=FIG_DPI,bbox_inches="tight"); plt.show(); plt.close(fig)

## 29. Historical vs future distribution

In [ ]:
def plot_distributions(baseline, future, title, xlabel, filename, bins=30):
    fig,ax=plt.subplots(figsize=(9,5)); ax.hist(np.asarray(baseline).ravel(),bins=bins,alpha=.55,label="Baseline"); ax.hist(np.asarray(future).ravel(),bins=bins,alpha=.55,label="Future")
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel("Frequency"); ax.legend(); fig.tight_layout(); fig.savefig(figure_path(filename),dpi=FIG_DPI,bbox_inches="tight"); plt.show(); plt.close(fig)

## 30. Boxplot comparison

In [ ]:
def plot_box(baseline, future, title, ylabel, filename):
    fig,ax=plt.subplots(figsize=(7,5)); ax.boxplot([np.asarray(baseline).ravel(),np.asarray(future).ravel()]); ax.set_xticks([1,2]); ax.set_xticklabels(["Baseline","Future"]); ax.set_ylabel(ylabel); ax.set_title(title)
    fig.tight_layout(); fig.savefig(figure_path(filename),dpi=FIG_DPI,bbox_inches="tight"); plt.show(); plt.close(fig)

## 31. Annual aggregation

In [ ]:
def annual_series(da, method="mean"):
    g=da.groupby("time.year")
    if method=="mean": return g.mean("time")
    if method=="sum": return g.sum("time")
    if method=="max": return g.max("time")
    if method=="min": return g.min("time")
    raise ValueError("method must be mean, sum, max or min")

## 32. Linear trend

In [ ]:
def linear_trend(annual):
    x=annual.year.values.astype(float); y=annual.values.astype(float); mask=np.isfinite(y)
    return np.polyfit(x[mask],y[mask],1)[0] if mask.sum()>=3 else np.nan

## 33. Trend plot

In [ ]:
def plot_trend(annual,title,ylabel,filename):
    slope=linear_trend(annual); x=annual.year.values.astype(float); y=annual.values.astype(float)
    fit=np.polyval(np.polyfit(x,y,1),x)
    fig,ax=plt.subplots(figsize=(10,5)); ax.plot(x,y,label="Annual value"); ax.plot(x,fit,linestyle="--",label=f"Trend = {slope:.3f}/year")
    ax.set_title(title); ax.set_xlabel("Year"); ax.set_ylabel(ylabel); ax.legend(); fig.tight_layout(); fig.savefig(figure_path(filename),dpi=FIG_DPI,bbox_inches="tight"); plt.show(); plt.close(fig)

## 34. Seasonal summaries

In [ ]:
def seasonal_series(da, method="mean"):
    g=da.groupby("time.season")
    if method=="mean": return g.mean("time")
    if method=="sum": return g.sum("time")
    if method=="max": return g.max("time")
    raise ValueError("method must be mean, sum or max")

## 35. Threshold exceedance frequency

In [ ]:
def threshold_frequency(da, threshold, direction="above"):
    mask = da > threshold if direction=="above" else da < threshold
    return mask.mean("time")*100

## 36. Threshold change

In [ ]:
def threshold_change(baseline,future,threshold,direction="above"):
    return threshold_frequency(future,threshold,direction)-threshold_frequency(baseline,threshold,direction)

## 37. Percentile summary

In [ ]:
def percentile_summary(da, percentiles=(5,25,50,75,95)):
    v=np.asarray(da).ravel(); return pd.DataFrame({"percentile":percentiles,"value":[np.nanpercentile(v,p) for p in percentiles]})

## 38. Tail frequency

In [ ]:
def upper_tail_frequency(da, threshold): return float(np.nanmean(np.asarray(da)>threshold)*100)
def lower_tail_frequency(da, threshold): return float(np.nanmean(np.asarray(da)<threshold)*100)

## 39. Multi-indicator summary

In [ ]:
def make_summary(items): return pd.DataFrame(items)
summary_rows=[]
# summary_rows.append(regional_summary("TXx",TXx_baseline,TXx_future))
# summary_rows.append(regional_summary("RX1day",RX1_baseline,RX1_future))
# summary_rows.append(regional_summary("RX5day",RX5_baseline,RX5_future))
summary_table=make_summary(summary_rows); summary_table

## 40. Indicator-change dashboard

In [ ]:
def plot_indicator_changes(df,filename):
    if df.empty: return
    fig,ax=plt.subplots(figsize=(10,6)); x=np.arange(len(df)); ax.bar(x,df["relative_change_percent"].values); ax.axhline(0,linewidth=1)
    ax.set_xticks(x); ax.set_xticklabels(df["indicator"],rotation=45,ha="right"); ax.set_ylabel("Relative change (%)"); ax.set_title("Projected change in climate indicators")
    fig.tight_layout(); fig.savefig(figure_path(filename),dpi=FIG_DPI,bbox_inches="tight"); plt.show(); plt.close(fig)

## 41. Scenario comparison

In [ ]:
def scenario_difference(a,b): return a-b
# Example: rcp85_minus_rcp26 = scenario_difference(future_rcp85,future_rcp26)

## 42. Risk map

In [ ]:
def risk_index(hazard,exposure,vulnerability): return hazard*exposure*vulnerability
def plot_risk(risk,title,filename): plot_map(risk,title,filename,cbar_label="Normalized risk index")

## 43. Risk-change map

In [ ]:
# risk_change = future_risk - baseline_risk
# plot_map(risk_change,"Projected risk change","risk_rcp26_change",cbar_label="Risk-index change")

## 44. Publication summary table

In [ ]:
def round_table(df,decimals=2):
    out=df.copy(); cols=out.select_dtypes(include=np.number).columns; out[cols]=out[cols].round(decimals); return out

## 45. Save tables

In [ ]:
def save_table(df,name):
    path=table_path(name); df.to_csv(path,index=False); print("Saved",path); return path

## 46. Save spatial result

In [ ]:
def save_netcdf(da,name,description=None):
    path=OUTPUTS/f"{name}.nc"
    if description: da.attrs["description"]=description
    da.to_netcdf(path); print("Saved",path); return path

## 47. Add metadata

In [ ]:
def add_metadata(da,indicator,scenario,baseline,future):
    da.attrs.update({"indicator":indicator,"scenario":scenario,"baseline_period":baseline,"future_period":future,"analysis_stage":"publication_reporting","created_by":"Notebook 08"})
    return da

## 48. Coordinate QC

In [ ]:
def coordinate_check(da):
    out={}
    for n in ["lat","latitude","lon","longitude"]:
        if n in da.coords:
            v=da[n].values; out[n]={"size":len(v),"first":float(v[0]),"last":float(v[-1]),"increasing":bool(np.all(np.diff(v)>0))}
    return out

## 49. Time QC

In [ ]:
def time_coverage(da): return {"start":str(da.time.values[0]),"end":str(da.time.values[-1]),"timesteps":da.sizes["time"]}
def duplicate_time_count(da): return int(len(da.time.values)-len(np.unique(da.time.values)))

## 50. Missing-data report

In [ ]:
def missing_data_report(da):
    total=da.size; missing=int(da.isnull().sum()); return {"total":total,"missing":missing,"missing_percent":missing/total*100 if total else np.nan}

## 51. Physical plausibility check

In [ ]:
def physical_range_check(da,minimum=None,maximum=None):
    v=np.asarray(da); result={"minimum":float(np.nanmin(v)),"maximum":float(np.nanmax(v))}
    if minimum is not None: result["below_minimum"]=int(np.sum(v<minimum))
    if maximum is not None: result["above_maximum"]=int(np.sum(v>maximum))
    return result

## 52. Model completeness

In [ ]:
def model_completeness(models):
    return pd.DataFrame({"model":[str(m) for m in models.model.values],"time_steps":[models.sel(model=m).sizes["time"] for m in models.model.values]})

## 53. Results inventory

In [ ]:
def inventory(folder,pattern="*"):
    files=sorted(Path(folder).glob(pattern)); return pd.DataFrame({"file":[f.name for f in files],"size_kb":[round(f.stat().st_size/1024,1) for f in files]})
print("Figures:"); display(inventory(FIGURES,"*.png")) if list(FIGURES.glob("*.png")) else print("No figures yet.")

## 54. Figure QC

In [ ]:
def figure_qc(path):
    p=Path(path); return {"exists":p.exists(),"non_empty":p.stat().st_size>0 if p.exists() else False,"extension":p.suffix.lower()}

## 55. Table QC

In [ ]:
def table_qc(df): return {"rows":len(df),"columns":len(df.columns),"missing":int(df.isna().sum().sum()),"duplicates":int(df.duplicated().sum())}

## 56. Common range for comparable maps

In [ ]:
def common_range(*arrays):
    v=np.concatenate([np.asarray(a).ravel() for a in arrays]); v=v[np.isfinite(v)]; return float(v.min()),float(v.max())

## 57. Symmetric anomaly range

In [ ]:
def symmetric_range(da):
    m=float(np.nanmax(np.abs(np.asarray(da)))); return -m,m

## 58. Publication map checklist

In [ ]:
MAP_CHECKLIST=["Correct coordinates","Correct units","Correct periods","Consistent extent","Appropriate color scale","Labelled colorbar","Missing values checked","High resolution","Descriptive filename"]
for x in MAP_CHECKLIST: print("[ ]",x)

## 59. Statistical checklist

In [ ]:
STATISTICAL_CHECKLIST=["Baseline/future periods verified","All six models included","No duplicate model","Ensemble dimension verified","Agreement threshold stated","Uncertainty metric stated","Trend units stated","Relative change checked for zero baseline","Threshold units verified"]
for x in STATISTICAL_CHECKLIST: print("[ ]",x)

## 60. Attribution language check
Do not write “climate change caused X% of the event” unless a formal attribution methodology supports that claim. For projection studies, prefer “projected change”, “climate-change signal”, “model agreement”, and “inter-model uncertainty”.

## 61. Model spread is not automatically probability
P10–P90, standard deviation, range and IQR describe **inter-model spread** unless a formal probabilistic framework justifies another interpretation.

## 62. Recommended main-paper figures
1. Study domain
2. Historical model evaluation
3. Future ensemble change
4. Model agreement
5. Inter-model uncertainty
6. Regional time series
7. Extreme-indicator changes
8. Hazard/risk change

## 63. Supplementary figures
Individual model maps, seasonal maps, alternative scenarios, sensitivity tests, additional indicators, model time series and additional uncertainty diagnostics can be moved to supplementary material.

## 64. Figure caption template
**Figure X.** Projected change in **[indicator]** under **[scenario]** for **[future period]** relative to **[baseline period]**. The map shows the multi-model ensemble **[mean/median]**. Hatching indicates regions where **[agreement criterion]** is satisfied. Values are shown in **[units]**.

## 65. Table caption template
**Table X.** Baseline values, future values and projected changes in selected climate indicators. Values represent the multi-model ensemble **[mean/median]** over **[region]**.

## 66. Uncertainty interpretation template
“The ensemble members generally agree on the direction of change, although the magnitude varies among models. The inter-model spread indicates uncertainty in the magnitude of the projected response.”

## 67. Risk interpretation template
“The projected hazard overlaps with areas of higher exposure and/or vulnerability, producing an increase in the normalized risk index. Because exposure and vulnerability representations introduce assumptions, the result should be interpreted as a comparative assessment rather than an exact prediction of losses.”

## 68. Final report structure
### Executive summary
### Data and methods
### Historical evaluation
### Projected climate change
### Model agreement and uncertainty
### Impact and risk
### Limitations
### Conclusions

## 69. Reproducibility record

In [ ]:
def reproducibility_record():
    return {"variable":VARIABLE,"scenario":SCENARIO,"baseline":f"{BASELINE_START} to {BASELINE_END}","future":f"{FUTURE_START} to {FUTURE_END}","figure_dpi":FIG_DPI}
reproducibility_record()

## 70. Results manifest

In [ ]:
def create_manifest():
    rows=[]
    for root in [FIGURES,TABLES,OUTPUTS,REPORTS]:
        for f in root.rglob("*") if root.exists() else []:
            if f.is_file(): rows.append({"category":root.name,"file":str(f.relative_to(BASE)),"size_kb":round(f.stat().st_size/1024,1)})
    return pd.DataFrame(rows)
manifest=create_manifest(); manifest.head()

## 71. Save manifest

In [ ]:
manifest_path=REPORTS/"results_manifest.csv"; manifest.to_csv(manifest_path,index=False); print("Saved",manifest_path)

## 72. Main-paper output package

In [ ]:
MAIN_PAPER_OUTPUTS=["study_domain.png","historical_evaluation.png","future_change.png","model_agreement.png","model_uncertainty.png","regional_timeseries.png","extreme_indicator_changes.png","risk_change.png"]
MAIN_PAPER_OUTPUTS

## 73. Final data QC

In [ ]:
FINAL_DATA_QC=["Input datasets documented","Units verified","Calendar/time axis verified","Missing values assessed","Physical ranges checked","Baseline verified","Future period verified","Six models accounted for","Ensemble verified","Agreement calculated","Uncertainty documented"]
for x in FINAL_DATA_QC: print("[ ]",x)

## 74. Final figure QC

In [ ]:
FINAL_FIGURE_QC=["Title correct","Units correct","Legend readable","Colorbar labelled","Scale appropriate","Geographic extent consistent","Resolution adequate","Filename descriptive","Caption prepared"]
for x in FINAL_FIGURE_QC: print("[ ]",x)

## 75. Final interpretation QC

In [ ]:
FINAL_INTERPRETATION_QC=["Magnitude reported","Direction reported","Spatial pattern discussed","Model agreement discussed","Uncertainty discussed","Limitations stated","Attribution claims match method","Risk claims are appropriately qualified"]
for x in FINAL_INTERPRETATION_QC: print("[ ]",x)

## 76. Complete repository workflow
**01 Data preprocessing → 02 Extremes/trends → 03–05 Indicators and specialized analyses → 06 Bias correction & ensemble → 07 Impact attribution/risk → 08 Visualization, uncertainty & reporting.**

## 77. Final recommendation
At this point, do not add another core notebook merely to extend the sequence. The next priority is to execute Notebooks 01–08 on the actual six-model dataset, validate the outputs, resolve data-specific issues, and generate the final research figures and tables.

## 78. Conclusion
Notebook 08 completes the reporting layer of the climate attribution toolkit. It provides reusable tools for maps, time series, model agreement, uncertainty, trends, thresholds, risk visualization, tables, metadata, QC and reproducibility. Formal event attribution, where required, should remain a separate statistical analysis with an explicitly defined factual/counterfactual framework.